In [1]:
import subprocess
subprocess.run(['pip', 'install', 'onnx', 'onnxruntime', '-q'], check=True)
print('Dependencies ready')

Dependencies ready


In [2]:
import os, time, copy, warnings, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.quantization as tq
import torchvision.models as models
import pandas as pd

warnings.filterwarnings('ignore')

# Reproducibility + CPU tuning
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CPU_THREADS = max(1, (os.cpu_count() or 1) - 1)
torch.set_num_threads(CPU_THREADS)
torch.set_num_interop_threads(1)

# Quantized ops require a quantization engine on CPU
if 'fbgemm' in torch.backends.quantized.supported_engines:
    torch.backends.quantized.engine = 'fbgemm'
else:
    torch.backends.quantized.engine = torch.backends.quantized.supported_engines[0]
print(f"Quant engine: {torch.backends.quantized.engine}")
print(f"CPU threads : {torch.get_num_threads()} (interop={torch.get_num_interop_threads()})")

# ✅ FIXED PATH — matches your Kaggle model location
MODEL_PATH  = '/kaggle/input/models/awaisverse/deepfakemodel/pytorch/default/1/model.pth'
OUTPUT_DIR  = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Change IMG_SIZE if your model uses a different input resolution
IMG_SIZE    = 224
DUMMY_INPUT = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

# Benchmark/Calibration knobs
BENCH_WARMUP = 10
BENCH_RUNS   = 100
N_CALIB      = 100
CALIB_BATCH  = 4

print(f'PyTorch {torch.__version__} | CPU cores: {os.cpu_count()}')
print(f'Model  : {MODEL_PATH}')
print(f'Output : {OUTPUT_DIR}')

# ✅ Auto-find the actual .pth file in case the filename differs
import glob
matches = glob.glob('/kaggle/input/models/awaisverse/deepfakemodel/pytorch/default/1/*.pth')
print(f'
Found .pth files: {matches}')
if matches and not os.path.exists(MODEL_PATH):
    MODEL_PATH = matches[0]
    print(f'Using: {MODEL_PATH}')



PyTorch 2.9.0+cu126 | CPU cores: 4
Model  : /kaggle/input/models/awaisverse/deepfakemodel/pytorch/default/1/model.pth
Output : /kaggle/working

Found .pth files: ['/kaggle/input/models/awaisverse/deepfakemodel/pytorch/default/1/model.pth']


In [3]:
def load_model(path: str) -> nn.Module:
    """
    Handles:
      - Full model saved with torch.save(model, ...)
      - Checkpoint dict with 'model' key
      - Plain state_dict (auto-detects architecture from key patterns)
    """
    ckpt = torch.load(path, map_location='cpu')
    print(f'Checkpoint type : {type(ckpt)}')

    # Case 1: full nn.Module
    if isinstance(ckpt, nn.Module):
        print('Loaded as: full model')
        return ckpt.eval()

    # Case 2: dict with a 'model' nn.Module
    if isinstance(ckpt, dict) and isinstance(ckpt.get('model'), nn.Module):
        print('Loaded as: checkpoint["model"]')
        return ckpt['model'].eval()

    # Case 3: state dict
    if isinstance(ckpt, dict):
        sd   = ckpt.get('state_dict', ckpt)
        arch = str(ckpt.get('arch', ckpt.get('architecture', ''))).lower()
        nc   = int(ckpt.get('num_classes', 2))
        keys = ' '.join(list(sd.keys())[:30]).lower()

        print(f'arch hint   : {arch or "(none)"}')
        print(f'num_classes : {nc}')
        print(f'Sample keys : {list(sd.keys())[:5]}')

        if 'xception' in arch or 'xception' in keys:
            import timm
            m = timm.create_model('xception', pretrained=False, num_classes=nc)
            print('Architecture: Xception (timm)')

        elif 'efficientnet_b0' in arch:
            m = models.efficientnet_b0(weights=None)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, nc)
            print('Architecture: EfficientNet-B0')

        elif 'efficientnet' in arch or 'efficientnet' in keys:
            m = models.efficientnet_b4(weights=None)
            m.classifier[1] = nn.Linear(m.classifier[1].in_features, nc)
            print('Architecture: EfficientNet-B4')

        elif 'resnet18' in arch:
            m = models.resnet18(weights=None)
            m.fc = nn.Linear(m.fc.in_features, nc)
            print('Architecture: ResNet-18')

        elif 'resnet34' in arch:
            m = models.resnet34(weights=None)
            m.fc = nn.Linear(m.fc.in_features, nc)
            print('Architecture: ResNet-34')

        elif 'resnet' in arch or ('layer' in keys and 'bn' in keys and 'downsample' in keys):
            m = models.resnet50(weights=None)
            m.fc = nn.Linear(m.fc.in_features, nc)
            print('Architecture: ResNet-50')

        elif 'vgg' in arch or 'features.0' in keys:
            m = models.vgg16(weights=None)
            m.classifier[6] = nn.Linear(4096, nc)
            print('Architecture: VGG-16')

        elif 'densenet' in arch or 'denseblock' in keys:
            m = models.densenet121(weights=None)
            m.classifier = nn.Linear(m.classifier.in_features, nc)
            print('Architecture: DenseNet-121')

        elif 'inception' in arch or 'inception' in keys:
            m = models.inception_v3(weights=None, aux_logits=False)
            m.fc = nn.Linear(m.fc.in_features, nc)
            print('Architecture: InceptionV3')

        elif 'mobile' in arch or 'mobile' in keys:
            m = models.mobilenet_v3_large(weights=None)
            m.classifier[3] = nn.Linear(m.classifier[3].in_features, nc)
            print('Architecture: MobileNetV3-Large')

        else:
            print('Architecture unknown — defaulting to ResNet-50')
            print('Edit this cell to specify your architecture if results look wrong.')
            m = models.resnet50(weights=None)
            m.fc = nn.Linear(m.fc.in_features, nc)

        # Strip DataParallel 'module.' prefix if present
        clean_sd = {k.replace('module.', ''): v for k, v in sd.items()}
        missing, unexpected = m.load_state_dict(clean_sd, strict=False)
        if missing:    print(f'Missing keys    : {missing[:3]}')
        if unexpected: print(f'Unexpected keys : {unexpected[:3]}')
        return m.eval()

    raise RuntimeError(f'Unrecognised checkpoint format: {type(ckpt)}')


model    = load_model(MODEL_PATH)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel  : {type(model).__name__}')
print(f'Params : {n_params:,}')

Checkpoint type : <class 'collections.OrderedDict'>
arch hint   : (none)
num_classes : 2
Sample keys : ['conv_stem.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var']
Architecture unknown — defaulting to ResNet-50
Edit this cell to specify your architecture if results look wrong.
Missing keys    : ['conv1.weight', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight']
Unexpected keys : ['conv_stem.weight', 'blocks.0.0.conv_dw.weight', 'blocks.0.0.bn1.weight']

Model  : ResNet
Params : 23,512,130


In [4]:
class QuantReadyModel(nn.Module):
    """Wrap a float model with QuantStub/DeQuantStub for eager static quantization."""
    def __init__(self, base_model: nn.Module):
        super().__init__()
        self.quant = tq.QuantStub()
        self.model = base_model
        self.dequant = tq.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.model(x)
        x = self.dequant(x)
        return x

def size_mb(path: str) -> float:
    return os.path.getsize(path) / 1e6

def benchmark(m, inp=None, n=None, warmup=None):
    if inp is None:
        inp = DUMMY_INPUT
    if n is None:
        n = BENCH_RUNS
    if warmup is None:
        warmup = BENCH_WARMUP

    m.eval()
    with torch.inference_mode():
        for _ in range(warmup):
            _ = m(inp)
        t0 = time.perf_counter()
        for _ in range(n):
            _ = m(inp)
        dt = (time.perf_counter() - t0) / n * 1000
    return dt

def benchmark_ort(sess, inp_name, np_inp, n=None, warmup=None):
    if n is None:
        n = BENCH_RUNS
    if warmup is None:
        warmup = BENCH_WARMUP
    for _ in range(warmup):
        sess.run(None, {inp_name: np_inp})
    t0 = time.perf_counter()
    for _ in range(n):
        sess.run(None, {inp_name: np_inp})
    return (time.perf_counter() - t0) / n * 1000

# Baseline
orig_path = f'{OUTPUT_DIR}/model_original_fp32.pth'
torch.save(model.state_dict(), orig_path)
orig_sz   = size_mb(orig_path)
orig_time = benchmark(model)

results = [{
    'Method': 'Original FP32',
    'Size_MB': round(orig_sz, 2),
    'Time_ms': round(orig_time, 2),
    'Size_Red_%': 0.0,
    'Speedup_x': 1.0
}]

print(f'Baseline | Size: {orig_sz:.2f} MB | Inference: {orig_time:.2f} ms')



Baseline | Size: 94.37 MB | Inference: 67.06 ms


In [5]:
from torch.quantization import quantize_dynamic

m_dyn = quantize_dynamic(
    copy.deepcopy(model),
    qconfig_spec={nn.Linear, nn.Conv2d, nn.LSTM, nn.GRU, nn.RNN},
    dtype=torch.qint8
)
m_dyn.eval()

dyn_path = f'{OUTPUT_DIR}/model_dynamic_int8.pth'
torch.save(m_dyn, dyn_path)

sz = size_mb(dyn_path)
t  = benchmark(m_dyn)
results.append({
    'Method': 'Dynamic INT8',
    'Size_MB': round(sz, 2),
    'Time_ms': round(t, 2),
    'Size_Red_%': round((1 - sz / orig_sz) * 100, 1),
    'Speedup_x': round(orig_time / t, 2)
})
print(f'Dynamic INT8 | {sz:.2f} MB | {t:.2f} ms | {results[-1]["Speedup_x"]}x speedup')

Dynamic INT8 | 94.38 MB | 71.52 ms | 0.94x speedup


In [6]:
# Replace torch.randn() with your real validation images for best accuracy
calib_data = [torch.randn(CALIB_BATCH, 3, IMG_SIZE, IMG_SIZE) for _ in range(max(1, N_CALIB // CALIB_BATCH))]

# Track whether FX static PTQ was already used as a fallback in this cell
FX_PTQ_DONE = False

m_ptq = QuantReadyModel(copy.deepcopy(model).eval()).eval()
m_ptq.qconfig = tq.get_default_qconfig(torch.backends.quantized.engine)

# Attempt Conv-BN-ReLU fusion
fused = False
for fuse_pattern in ([['model.conv1', 'model.bn1', 'model.relu']], [['conv1', 'bn1', 'relu']]):
    try:
        m_ptq = tq.fuse_modules(m_ptq, fuse_pattern, inplace=False)
        print(f'Layer fusion applied: {fuse_pattern[0]}')
        fused = True
        break
    except Exception:
        pass
if not fused:
    print('Fusion skipped (non-critical): compatible Conv-BN-ReLU pattern not found')

m_ptq = tq.prepare(m_ptq, inplace=False)

print(f'Calibrating (~{N_CALIB} samples, batch={CALIB_BATCH})...')
with torch.no_grad():
    for i, x in enumerate(calib_data):
        m_ptq(x)
        if (i + 1) % max(1, len(calib_data)//4) == 0:
            done = min((i+1) * CALIB_BATCH, N_CALIB)
            print(f'  {done}/{N_CALIB}')

m_ptq = tq.convert(m_ptq, inplace=False).eval()

# Some architectures (e.g., standard torchvision ResNet blocks) require FloatFunctional
# rewrites for eager-mode residual adds. If eager PTQ fails at runtime, fall back to FX PTQ.
ptq_method = 'Static PTQ INT8'
try:
    with torch.no_grad():
        _ = m_ptq(DUMMY_INPUT)
except NotImplementedError as e:
    print(f'Eager PTQ runtime not supported for this architecture: {e}')
    print('Falling back to FX Graph static PTQ for compatibility...')

    from torch.quantization.quantize_fx import prepare_fx, convert_fx
    from torch.ao.quantization import get_default_qconfig_mapping

    m_fx_fb = copy.deepcopy(model).eval()
    qmap_fb = get_default_qconfig_mapping(torch.backends.quantized.engine)
    m_fx_fb = prepare_fx(m_fx_fb, qmap_fb, (DUMMY_INPUT,))

    with torch.no_grad():
        for x in calib_data:
            m_fx_fb(x)

    m_ptq = convert_fx(m_fx_fb).eval()
    ptq_method = 'Static PTQ INT8 (FX fallback)'
    FX_PTQ_DONE = True

ptq_path = f'{OUTPUT_DIR}/model_static_ptq_int8.pth'
torch.save(m_ptq, ptq_path)

sz = size_mb(ptq_path)
t  = benchmark(m_ptq)
results.append({
    'Method': ptq_method,
    'Size_MB': round(sz, 2),
    'Time_ms': round(t, 2),
    'Size_Red_%': round((1 - sz / orig_sz) * 100, 1),
    'Speedup_x': round(orig_time / t, 2)
})
print(f'{ptq_method:<16} | {sz:.2f} MB | {t:.2f} ms | {results[-1]["Speedup_x"]}x speedup')



Layer fusion applied
Calibrating (100 samples)...
  25/100
  50/100
  75/100
  100/100


NotImplementedError: Could not run 'quantized::conv2d_relu.new' with arguments from the 'CPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::conv2d_relu.new' is only available for these backends: [Meta, QuantizedCPU, QuantizedCUDA, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradMeta, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFrontMode, PreDispa

In [ ]:
try:
    if globals().get('FX_PTQ_DONE', False):
        print('FX quantization skipped: already executed as static PTQ fallback above')
    else:
        from torch.quantization.quantize_fx import prepare_fx, convert_fx
        from torch.ao.quantization import get_default_qconfig_mapping

        m_fx = copy.deepcopy(model).eval()
        qmap = get_default_qconfig_mapping(torch.backends.quantized.engine)
        m_fx = prepare_fx(m_fx, qmap, (DUMMY_INPUT,))

        print('Calibrating FX model...')
        with torch.no_grad():
            for x in calib_data:
                m_fx(x)

        m_fx = convert_fx(m_fx).eval()

        fx_path = f'{OUTPUT_DIR}/model_fx_graph_int8.pth'
        torch.save(m_fx, fx_path)

        sz = size_mb(fx_path)
        t  = benchmark(m_fx)
        results.append({
            'Method': 'FX Graph INT8',
            'Size_MB': round(sz, 2),
            'Time_ms': round(t, 2),
            'Size_Red_%': round((1 - sz / orig_sz) * 100, 1),
            'Speedup_x': round(orig_time / t, 2)
        })
        print(f'FX Graph INT8| {sz:.2f} MB | {t:.2f} ms | {results[-1]["Speedup_x"]}x speedup')

except Exception as e:
    print(f'FX quantization skipped: {e}')



In [ ]:
m_fp16 = copy.deepcopy(model).half().eval()

fp16_path = f'{OUTPUT_DIR}/model_fp16.pth'
torch.save(m_fp16, fp16_path)

sz = size_mb(fp16_path)
try:
    t = benchmark(m_fp16, DUMMY_INPUT.half())
except Exception:
    t = float('nan')
    print('FP16 native CPU inference not supported; model saved (half the size)')

results.append({
    'Method': 'FP16',
    'Size_MB': round(sz, 2),
    'Time_ms': round(t, 2) if t == t else 'N/A',
    'Size_Red_%': round((1 - sz / orig_sz) * 100, 1),
    'Speedup_x': round(orig_time / t, 2) if t == t else 'N/A'
})
print(f'FP16         | {sz:.2f} MB')

In [ ]:
try:
    m_ts = copy.deepcopy(m_dyn).eval()
    with torch.no_grad():
        traced = torch.jit.trace(m_ts, DUMMY_INPUT)
    traced = torch.jit.optimize_for_inference(traced)

    ts_path = f'{OUTPUT_DIR}/model_torchscript_int8.pt'
    traced.save(ts_path)

    sz = size_mb(ts_path)
    t  = benchmark(traced)
    results.append({
        'Method': 'TorchScript INT8',
        'Size_MB': round(sz, 2),
        'Time_ms': round(t, 2),
        'Size_Red_%': round((1 - sz / orig_sz) * 100, 1),
        'Speedup_x': round(orig_time / t, 2)
    })
    print(f'TorchScript  | {sz:.2f} MB | {t:.2f} ms | {results[-1]["Speedup_x"]}x speedup')

except Exception as e:
    print(f'TorchScript skipped: {e}')

In [ ]:
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic as onnx_qdyn, QuantType

# Export FP32 ONNX
onnx_fp32 = f'{OUTPUT_DIR}/model_fp32.onnx'
with torch.no_grad():
    torch.onnx.export(
        copy.deepcopy(model).eval(),
        DUMMY_INPUT,
        onnx_fp32,
        opset_version=17,
        export_params=True,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
    )
onnx.checker.check_model(onnx.load(onnx_fp32))
print(f'ONNX FP32 exported: {size_mb(onnx_fp32):.2f} MB')

# Quantize to INT8
onnx_int8 = f'{OUTPUT_DIR}/model_onnx_int8.onnx'
onnx_qdyn(
    onnx_fp32, onnx_int8,
    weight_type=QuantType.QInt8,
    optimize_model=True,
    per_channel=True,
    reduce_range=True
)

# Benchmark
sess_opt = ort.SessionOptions()
sess_opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_opt.intra_op_num_threads = os.cpu_count()
sess     = ort.InferenceSession(onnx_int8, sess_opt, providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name
np_inp   = DUMMY_INPUT.numpy()

onnx_t = benchmark_ort(sess, inp_name, np_inp)

sz = size_mb(onnx_int8)
results.append({
    'Method': 'ONNX INT8',
    'Size_MB': round(sz, 2),
    'Time_ms': round(onnx_t, 2),
    'Size_Red_%': round((1 - sz / orig_sz) * 100, 1),
    'Speedup_x': round(orig_time / onnx_t, 2)
})
print(f'ONNX INT8    | {sz:.2f} MB | {onnx_t:.2f} ms | {results[-1]["Speedup_x"]}x speedup')

In [ ]:
# Replace torch.randn() with your real training data for production use
try:
    m_qat = QuantReadyModel(copy.deepcopy(model))
    m_qat.train()
    m_qat.qconfig = tq.get_default_qat_qconfig(torch.backends.quantized.engine)
    m_qat = tq.prepare_qat(m_qat, inplace=False)

    optimizer = torch.optim.SGD(m_qat.parameters(), lr=1e-5, momentum=0.9)
    criterion = nn.CrossEntropyLoss()

    QAT_STEPS = 50
    print(f'QAT fine-tuning for {QAT_STEPS} steps (synthetic data)')
    print('Replace synthetic data with real validation images for production!')

    m_qat.train()
    for step in range(QAT_STEPS):
        x    = torch.randn(4, 3, IMG_SIZE, IMG_SIZE)
        y    = torch.randint(0, 2, (4,))
        optimizer.zero_grad()
        out  = m_qat(x)
        if isinstance(out, tuple): out = out[0]
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        if (step + 1) % 10 == 0:
            print(f'  [{step+1}/{QAT_STEPS}] loss={loss.item():.4f}')

    m_qat.eval()
    m_qat = tq.convert(m_qat, inplace=False).eval()

    qat_path = f'{OUTPUT_DIR}/model_qat_int8.pth'
    torch.save(m_qat, qat_path)

    sz = size_mb(qat_path)
    t  = benchmark(m_qat)
    results.append({
        'Method': 'QAT INT8',
        'Size_MB': round(sz, 2),
        'Time_ms': round(t, 2),
        'Size_Red_%': round((1 - sz / orig_sz) * 100, 1),
        'Speedup_x': round(orig_time / t, 2)
    })
    print(f'QAT INT8     | {sz:.2f} MB | {t:.2f} ms | {results[-1]["Speedup_x"]}x speedup')

except Exception as e:
    print(f'QAT skipped: {e}')

In [ ]:
df = pd.DataFrame(results).sort_values('Time_ms', ascending=True, na_position='last').reset_index(drop=True)
print('=' * 72)
print('QUANTIZATION RESULTS SUMMARY (sorted by latency)')
print('=' * 72)
print(df.to_string(index=False))
print('=' * 72)

df.to_csv(f'{OUTPUT_DIR}/quantization_summary.csv', index=False)
best = df.iloc[0]['Method'] if len(df) else 'N/A'
print(f'
Saved: quantization_summary.csv | Best latency: {best}')

# Artifact manifest for easy deployment decisions
artifacts = []
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp):
        artifacts.append({'file': f, 'size_mb': round(size_mb(fp), 2)})

manifest = {
    'model_path': MODEL_PATH,
    'img_size': IMG_SIZE,
    'quant_engine': torch.backends.quantized.engine,
    'cpu_threads': torch.get_num_threads(),
    'bench_warmup': BENCH_WARMUP,
    'bench_runs': BENCH_RUNS,
    'artifacts': artifacts,
}

with open(f'{OUTPUT_DIR}/artifact_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('
Output files:')
for a in artifacts:
    print(f"  {a['file']:<48} {a['size_mb']:>8.2f} MB")
print('Saved: artifact_manifest.json')



In [ ]:
if len(df):
    fastest = df.iloc[0]['Method']
    smallest = df.sort_values('Size_MB').iloc[0]['Method']
else:
    fastest, smallest = 'N/A', 'N/A'

print(f"""
RECOMMENDATIONS
==============================================================
 Fastest on this CPU        ->  {fastest}
 Smallest model file        ->  {smallest}
 Recommended default        ->  ONNX INT8 (if ORT available)
 Pure PyTorch fallback      ->  Dynamic INT8 (.pth)
==============================================================

HOW TO PICK A MODEL FILE
1) Prefer ONNX INT8 for CPU-only end users (best portability/perf).
2) If you deploy only with PyTorch, start with model_dynamic_int8.pth.
3) Use model_torchscript_int8.pt when you want a single optimized TorchScript artifact.
4) Use model_static_ptq_int8.pth when eager PTQ worked; if fallback triggered, that file is FX-quantized.
5) Keep model_original_fp32.pth as compatibility fallback.

HOW TO USE — ONNX (recommended):
    import onnxruntime as ort, numpy as np
    sess = ort.InferenceSession('model_onnx_int8.onnx', providers=['CPUExecutionProvider'])
    logits = sess.run(None, {{'input': image_np}})[0]

HOW TO USE — Pure PyTorch:
    model = torch.load('model_dynamic_int8.pth', map_location='cpu')
    model.eval()
    with torch.no_grad():
        logits = model(image_tensor.unsqueeze(0))

See artifact_manifest.json + quantization_summary.csv for exact outputs and best method.
""")



In [ ]:
model.eval()
m_dyn.eval()
test_x = torch.randn(10, 3, IMG_SIZE, IMG_SIZE)

with torch.no_grad():
    p_orig = torch.softmax(model(test_x), 1)
    p_dyn  = torch.softmax(m_dyn(test_x), 1)

mae       = (p_orig - p_dyn).abs().mean().item()
agreement = (p_orig.argmax(1) == p_dyn.argmax(1)).float().mean().item()

print('Accuracy check — Original vs Dynamic INT8:')
print(f'  Mean probability difference : {mae:.5f}  (target < 0.01 = excellent)')
print(f'  Prediction agreement        : {agreement*100:.1f}%')
print('\nAll models saved to /kaggle/working/')